# Complete Evaluation: NMSE + Outage + Multi-SNR + Budget Allocation

하나의 노트북에서 모든 실험 결과를 재생성합니다.

1. **Cell 1**: 환경 셋업
2. **Cell 2**: MT-AE: Multi-SNR (10/20/30) outage curves + NMSE + budget allocation
3. **Cell 3**: CLNet/CRNet: outage curves + NMSE + budget allocation (SNR=20)
4. **Cell 4**: Figure 생성 (전부)

## Cell 1: Environment Setup

In [ ]:
import os, sys

PROJECT_ROOT = '/content/drive/MyDrive/MambaCompression'
MAMBAIC_ROOT = os.path.join(PROJECT_ROOT, 'MambaIC')

if not os.path.isdir(PROJECT_ROOT):
    from google.colab import drive
    drive.mount('/content/drive')

os.chdir(MAMBAIC_ROOT)
if MAMBAIC_ROOT not in sys.path:
    sys.path.insert(0, MAMBAIC_ROOT)

setup_path = os.path.join(PROJECT_ROOT, 'setup_colab.py')
if os.path.isfile(setup_path):
    exec(open(setup_path).read())
else:
    !pip install -q einops scipy tqdm thop fvcore pybind11
!pip install -q seaborn compressai timm pulp 2>/dev/null | tail -1

import torch
print(f'CUDA: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
print('Ready.')

## Cell 2: MT-AE Multi-SNR Outage + NMSE + Budget Allocation

SNR=10, 20, 30에서 per-bin outage curves + NMSE를 측정하고, budget allocation 최적화.

**예상 시간: ~15분** (3 SNR × 49 saving levels × inference)

In [ ]:
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from matplotlib.ticker import MultipleLocator
import numpy as np
import pandas as pd
import os

MAMBAIC_ROOT = '/content/drive/MyDrive/MambaCompression/MambaIC'
os.chdir(MAMBAIC_ROOT)
RESULTS_CSV = os.path.join(MAMBAIC_ROOT, 'results', 'csv')
FIGURES_DIR = os.path.join(MAMBAIC_ROOT, 'results', 'plots')
PAPER_FIG_DIR = os.path.join(os.path.dirname(MAMBAIC_ROOT), 'figures')

STYLE = {
    'font.size': 13, 'axes.labelsize': 13, 'axes.titlesize': 13,
    'xtick.labelsize': 12, 'ytick.labelsize': 12, 'legend.fontsize': 11,
    'lines.linewidth': 2.0, 'lines.markersize': 7, 'figure.dpi': 150, 'savefig.dpi': 300,
}
C_BLUE, C_RED = '#1f77b4', '#d62728'

def _monotone_nondecreasing(y):
    y = np.array(y, dtype=float)
    for i in range(1, len(y)):
        if y[i] < y[i-1]: y[i] = y[i-1]
    return y

def _save(fig, name):
    for d in [FIGURES_DIR, PAPER_FIG_DIR]:
        for ext in ('pdf', 'png'):
            fig.savefig(os.path.join(d, f'{name}.{ext}'), dpi=300, bbox_inches='tight')
    plt.close(fig)

def _grid(ax):
    ax.grid(True, linestyle='--', alpha=0.4, color='#cccccc', zorder=0)

# --- Fig: Multi-SNR Budget Allocation (MT-AE) ---
csv_path = os.path.join(RESULTS_CSV, 'complete_eval_mtae.csv')
if os.path.exists(csv_path):
    df = pd.read_csv(csv_path)
    gamma = 0.95
    
    with plt.rc_context(STYLE):
        fig, axes = plt.subplots(1, 3, figsize=(15, 4), sharey=True)
        for idx, snr in enumerate([10, 20, 30]):
            ax = axes[idx]
            sub = df[(df['snr']==snr) & (df['gamma']==gamma)].sort_values('target_saving')
            x = sub['target_saving'].values
            y_eq = _monotone_nondecreasing(sub['outage_equal'].values)
            y_al = _monotone_nondecreasing(sub['outage_alloc'].values)
            
            ax.plot(x, y_eq, '--', color=C_RED, drawstyle='steps-post',
                    linewidth=2.0, label='Equal allocation')
            ax.plot(x, y_al, '-', color=C_BLUE, drawstyle='steps-post',
                    linewidth=2.0, label='Optimal allocation')
            
            labels = ['(a)', '(b)', '(c)']
            ax.set_title(f'{labels[idx]} SNR = {snr} dB', fontsize=13)
            ax.set_xlabel('BOPs Saving vs. FP32 (%)', fontsize=13)
            ax.set_xlim(84, 97)
            ax.xaxis.set_major_locator(MultipleLocator(2))
            ax.set_ylim(-0.02, 1.05)
            _grid(ax)
            if idx == 0:
                ax.set_ylabel('Outage Probability', fontsize=13)
                ax.legend(loc='upper left', fontsize=10, framealpha=0.9)
        
        fig.tight_layout(w_pad=2.0)
        _save(fig, 'fig_budget_alloc_multi_snr')
        print('Saved: fig_budget_alloc_multi_snr')
    
    # --- Fig: NMSE comparison ---
    with plt.rc_context(STYLE):
        fig, ax = plt.subplots(figsize=(8, 5))
        for snr, ls, label in [(10, '--', 'SNR=10'), (20, '-', 'SNR=20'), (30, '-.', 'SNR=30')]:
            sub = df[(df['snr']==snr) & (df['gamma']==0.95)].sort_values('target_saving')
            ax.plot(sub['target_saving'], sub['nmse_db'],
                    ls, color=C_BLUE if snr==20 else (C_RED if snr==10 else '#2ca02c'),
                    linewidth=2.0, drawstyle='steps-post', label=label)
        ax.set_xlabel('BOPs Saving vs. FP32 (%)', fontsize=13)
        ax.set_ylabel('NMSE (dB)', fontsize=13)
        ax.set_xlim(84, 97)
        ax.xaxis.set_major_locator(MultipleLocator(2))
        ax.legend(fontsize=10, framealpha=0.9)
        _grid(ax)
        fig.tight_layout()
        _save(fig, 'fig_nmse_vs_saving_alloc')
        print('Saved: fig_nmse_vs_saving_alloc')
else:
    print(f'Run Cell 2 first: {csv_path} not found')

# --- Regenerate existing paper figures ---
print('\nRegenerating all paper figures...')
!cd /content/drive/MyDrive/MambaCompression/MambaIC && python analysis/generate_paper_figures.py

## Cell 3: CLNet/CRNet Outage + NMSE + Budget Allocation (SNR=20)

**예상 시간: ~5분** (2 models × 49 saving levels)

In [ ]:
# Reuse joint_dp_outage.ipynb Cell 8 — already has kappa fix
# Just reload and run
import importlib
# The Cell 8 code is embedded in the notebook, so just run it from there.
# OR: run the baselines from the existing outage_alloc_baselines.csv if already computed.

baseline_csv = os.path.join(RESULTS_CSV, 'outage_alloc_baselines.csv')
if os.path.exists(baseline_csv):
    df_bl = pd.read_csv(baseline_csv)
    print(f'Loaded existing baselines: {len(df_bl)} rows')
    for model in df_bl['model'].unique():
        for gamma in [0.95]:
            sub = df_bl[(df_bl['model']==model) & (df_bl['gamma']==gamma) & (df_bl['improvement']>0.001)]
            if len(sub) > 0:
                best = sub.loc[sub['improvement_pct'].idxmax()]
                print(f'  {model} g={gamma}: {best["target_saving"]:.1f}%  '
                      f'equal={best["outage_equal"]:.4f} -> alloc={best["outage_alloc"]:.4f}')
else:
    print('Run joint_dp_outage.ipynb Cell 8 first.')

## Cell 4: Generate All Paper Figures

기존 figure + 새 figure (NMSE 비교, multi-SNR budget allocation) 전부 재생성.

In [ ]:
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from matplotlib.ticker import MultipleLocator
import numpy as np
import pandas as pd
import os

MAMBAIC_ROOT = '/content/drive/MyDrive/MambaCompression/MambaIC'
os.chdir(MAMBAIC_ROOT)
RESULTS_CSV = os.path.join(MAMBAIC_ROOT, 'results', 'csv')
FIGURES_DIR = os.path.join(MAMBAIC_ROOT, 'results', 'plots')
PAPER_FIG_DIR = os.path.join(os.path.dirname(MAMBAIC_ROOT), 'figures')

STYLE = {
    'font.size': 13, 'axes.labelsize': 13, 'axes.titlesize': 13,
    'xtick.labelsize': 12, 'ytick.labelsize': 12, 'legend.fontsize': 11,
    'lines.linewidth': 2.0, 'lines.markersize': 7, 'figure.dpi': 150, 'savefig.dpi': 300,
}
C_BLUE, C_RED = '#1f77b4', '#d62728'

def _monotone_nondecreasing(y):
    y = np.array(y, dtype=float)
    for i in range(1, len(y)):
        if y[i] < y[i-1]: y[i] = y[i-1]
    return y

def _save(fig, name):
    for d in [FIGURES_DIR, PAPER_FIG_DIR]:
        for ext in ('pdf', 'png'):
            fig.savefig(os.path.join(d, f'{name}.{ext}'), dpi=300, bbox_inches='tight')
    plt.close(fig)

def _grid(ax):
    ax.grid(True, linestyle='--', alpha=0.4, color='#cccccc', zorder=0)

# --- Fig: Multi-SNR Budget Allocation (MT-AE) ---
csv_path = os.path.join(RESULTS_CSV, 'complete_eval_mtae.csv')
if os.path.exists(csv_path):
    df = pd.read_csv(csv_path)
    gamma = 0.95
    
    with plt.rc_context(STYLE):
        fig, axes = plt.subplots(1, 3, figsize=(15, 4), sharey=True)
        for idx, snr in enumerate([10, 20, 30]):
            ax = axes[idx]
            sub = df[(df['snr']==snr) & (df['gamma']==gamma)].sort_values('target_saving')
            x = sub['target_saving'].values
            y_eq = _monotone_nondecreasing(sub['outage_equal'].values)
            y_al = _monotone_nondecreasing(sub['outage_alloc'].values)
            
            ax.plot(x, y_eq, '--', color=C_RED, drawstyle='steps-post',
                    linewidth=2.0, label='Equal allocation')
            ax.plot(x, y_al, '-', color=C_BLUE, drawstyle='steps-post',
                    linewidth=2.0, label='Optimal allocation')
            
            labels = ['(a)', '(b)', '(c)']
            ax.set_title(f'{labels[idx]} SNR = {snr} dB', fontsize=13)
            ax.set_xlabel('BOPs Saving vs. FP32 (%)', fontsize=13)
            ax.set_xlim(84, 97)
            ax.xaxis.set_major_locator(MultipleLocator(2))
            ax.set_ylim(-0.02, 1.05)
            _grid(ax)
            if idx == 0:
                ax.set_ylabel('Outage Probability', fontsize=13)
                ax.legend(loc='upper left', fontsize=10, framealpha=0.9)
        
        fig.tight_layout(w_pad=2.0)
        _save(fig, 'fig_budget_alloc_multi_snr')
        print('Saved: fig_budget_alloc_multi_snr')
    
    # --- Fig: NMSE comparison (equal vs alloc should be same) ---
    with plt.rc_context(STYLE):
        fig, ax = plt.subplots(figsize=(8, 5))
        for snr, ls, label in [(10, '--', 'SNR=10'), (20, '-', 'SNR=20'), (30, '-.', 'SNR=30')]:
            sub = df[(df['snr']==snr) & (df['gamma']==0.95)].sort_values('target_saving')
            ax.plot(sub['target_saving'], sub['nmse_db'],
                    ls, color=C_BLUE if snr==20 else (C_RED if snr==10 else '#2ca02c'),
                    linewidth=2.0, drawstyle='steps-post', label=label)
        ax.set_xlabel('BOPs Saving vs. FP32 (%)', fontsize=13)
        ax.set_ylabel('NMSE (dB)', fontsize=13)
        ax.set_title('(a) NMSE under segment-level MPQ', fontsize=13)
        ax.set_xlim(84, 97)
        ax.xaxis.set_major_locator(MultipleLocator(2))
        ax.legend(fontsize=10, framealpha=0.9)
        _grid(ax)
        fig.tight_layout()
        _save(fig, 'fig_nmse_vs_saving_alloc')
        print('Saved: fig_nmse_vs_saving_alloc')
else:
    print(f'Run Cell 2 first: {csv_path} not found')

# --- Regenerate all existing paper figures ---
print('\nRegenerating all paper figures...')
exec(open('analysis/generate_paper_figures.py').read())